# Ch01 - 强化学习入门与环境交互循环

理解 Agent-Environment 交互的基本范式，运行第一个 RL 程序。

> 本 Notebook 配套《强化学习全面教程》PDF 使用。运行前请先执行 `pip install torch gymnasium numpy matplotlib tqdm`。

In [2]:
# 本单元导入本教程通用工具与第三方库
import numpy as np
import torch
import gymnasium as gym
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# 中文字体配置（Linux 路径，macOS 用户请改用 Heiti SC 或 PingFang SC）
import os, platform
if platform.system() == 'Linux' and os.path.exists('/usr/share/fonts/truetype/chinese/NotoSansSC-Regular.ttf'):
    fm.fontManager.addfont('/usr/share/fonts/truetype/chinese/NotoSansSC-Regular.ttf')
plt.rcParams['font.sans-serif'] = ['Noto Sans SC', 'DejaVu Sans', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

# PyTorch 设备选择
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch 版本: {torch.__version__}')
print(f'使用设备: {device}')
print(f'Gymnasium 版本: {gym.__version__}')


PyTorch 版本: 2.13.0+cpu
使用设备: cpu
Gymnasium 版本: 1.3.0


## 1.1 强化学习的核心思想

强化学习（RL）通过 **智能体（Agent）** 与 **环境（Environment）** 反复交互，根据环境反馈的 **奖励（Reward）** 调整 **策略（Policy）**，目标是最大化长期累积奖励。

**核心循环**：
1. 观察状态 $s_t$
2. 依据策略 $\pi$ 选择动作 $a_t$
3. 环境转移到 $s_{t+1}$，返回奖励 $r_{t+1}$
4. 更新策略，回到步骤 1

本 Notebook 用 CartPole-v1 演示这个循环。

## 1.2 CartPole-v1：RL 的 Hello World

CartPole 是经典控制任务：一根杆子立在小车上，目标是通过左右移动小车让杆子保持直立。

- **状态**：4 维（小车位置、速度、杆子角度、角速度）
- **动作**：2 个（向左、向右）
- **奖励**：每步 +1，直到杆倒或达到 500 步上限

In [ ]:
env = gym.make('CartPole-v1')

# 观察状态与动作空间
print(f'观测空间: {env.observation_space}')
print(f'  形状: {env.observation_space.shape}')
print(f'  范围: [{env.observation_space.low}, {env.observation_space.high}]')
print(f'动作空间: {env.action_space}')
print(f'  动作数: {env.action_space.n}')

观测空间: Box([-4.8               -inf -0.41887903        -inf], [4.8               inf 0.41887903        inf], (4,), float32)
  形状: (4,)
  范围: [[-4.8               -inf -0.41887903        -inf], [4.8               inf 0.41887903        inf]]
动作空间: Discrete(2)
  动作数: 2


## 1.3 第一个 RL 程序：随机策略

用最简单的随机策略验证 Gymnasium API。注意 Gymnasium 0.26+ 的 API 变化：
- `reset()` 返回 `(obs, info)` 元组
- `step()` 返回 5 元组 `(obs, reward, terminated, truncated, info)`

In [ ]:
def random_policy(observation):
    """随机策略：均匀采样动作。"""
    return np.random.randint(0, 2)

def run_episode(env, policy, render=False):
    """运行一个完整回合，返回累计奖励。"""
    obs, _ = env.reset(seed=42)
    total_reward = 0.0
    terminated, truncated = False, False
    while not (terminated or truncated):
        action = policy(obs)
        obs, reward, terminated, truncated, _ = env.step(action)
        total_reward += reward
    return total_reward

# 单回合
r = run_episode(env, random_policy)
print(f'随机策略单回合奖励: {r}')

# 多回合统计
rewards = [run_episode(env, random_policy) for _ in range(20)]
print(f'20回合平均: {np.mean(rewards):.2f} ± {np.std(rewards):.2f}')
print(f'最大/最小: {max(rewards):.0f} / {min(rewards):.0f}')

随机策略单回合奖励: 23.0
20回合平均: 21.35 ± 15.19
最大/最小: 68 / 9


## 1.4 可视化随机策略效果

用动画展示随机策略在 CartPole 上的表现（对比后续章节训练后的效果）。

In [ ]:
from viz_utils import visualize_cartpole
visualize_cartpole(lambda s: np.random.randint(0, 2), title='随机策略 (训练前)',max_steps=200)

## 1.5 小结

随机策略在 CartPole 上只能拿 10-30 分；后续章节学完 DQN/PPO 后可达 500 分。

**关键收获**：
- 理解 Agent-Env 交互循环
- 掌握 Gymnasium 5 元组 API
- 了解 CartPole 状态/动作语义

**下一步**：第二章将引入 MDP 的形式化定义与贝尔曼方程。